# 🚀 Lab 26: Interactive Visuals with Bokeh

### 📘 Lab Overview
In this lab, you will learn how to build **interactive web-based visualizations using Bokeh**. You will create interactive plots for stock and weather data, add hover tooltips, implement sliders and filters, apply color mapping and legends, and export your work as standalone HTML files.

This lab has been adapted for **Google Colab**, so everything runs inside notebook cells. You will still generate HTML outputs for sharing, just like a professional data science workflow.

## 🎯 Objectives
* Understand the fundamentals of Bokeh for interactive web-based visualizations.
* Install and configure Bokeh in a Python environment.
* Create interactive plots with stock and weather data.
* Add hover tooltips to display contextual information.
* Implement interactive filters, sliders, and selectors using `CustomJS`.
* Use color legends and color mapping for better interpretation.
* Generate standalone HTML files for web deployment.

## 🧰 Prerequisites
* Basic knowledge of Python programming.
* Familiarity with pandas for data manipulation.
* Understanding of basic visualization concepts.

## 💡 What You Are Going to Do (ELI10)
In simple words, this lab teaches you how to make charts that users can interact with. Instead of just looking at a picture, you can:
* **Hover** over points to see hidden numbers.
* **Zoom and Pan** to look closer at parts of the data.
* **Filter** data using sliders and dropdown menus.
* **Color Code** information to make it easier to read.
* **Save** your work as a webpage (HTML) that anyone can open in a browser.

## ⚙️ Environment Setup
### 👶 ELI10: Setting Up the Tools
Before we start building, we need to make sure our computer (Colab) has the right 'tools' installed. We are installing `bokeh` for the charts and `pandas` for handling our data tables.

In [1]:
# Step 1: Install required libraries
# We use %pip to ensure the libraries are available in the Colab environment
%pip install bokeh pandas numpy requests matplotlib

# Step 2: Initialize Bokeh for notebook display
# This is critical in Colab to ensure plots show up inside the cell output
from bokeh.io import output_notebook
output_notebook()

print("Environment is ready and Bokeh is initialized.")

Environment is ready and Bokeh is initialized.


# Task 1: Environment Setup and Bokeh Installation

## Subtask 1.1 & 1.2: Verify and Test
Let's verify our Python version and run a quick test to ensure Bokeh can render a simple plot.

In [2]:
import sys
from bokeh.plotting import figure, show
from bokeh.io import output_file

# Print Python version for environment verification
print("Python version:", sys.version)

# Create a simple test plot
# We define a figure and add a scatter glyph
p = figure(title="Bokeh Installation Test", width=400, height=400)
p.scatter([1, 2, 3, 4, 5], [6, 7, 2, 4, 5], size=12, color="navy", alpha=0.5)

# Save as standalone HTML (this will appear in your Colab file browser)
output_file("test_bokeh_installation.html")

# Display the plot in the notebook
show(p)

print("If you see a scatter plot above, Bokeh is working correctly.")

Python version: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]


If you see a scatter plot above, Bokeh is working correctly.


# Task 2: Creating Basic Interactive Plots
### 👶 ELI10: Your First Interactive Chart
Think of a standard chart as a drawing on paper. A Bokeh chart is like a drawing on a tablet—you can use your fingers (or mouse) to move it around and zoom in on interesting spots.

In [3]:
import numpy as np
from bokeh.plotting import figure, show, output_file

# Generate sample sine wave data using numpy
x = np.linspace(0, 4 * np.pi, 100)
y = np.sin(x)

# Create figure with specific interactive tools
# 'pan' allows dragging, 'wheel_zoom' uses the mouse wheel
p = figure(
    title="Basic Interactive Plot",
    width=800,
    height=400,
    tools="pan,wheel_zoom,box_zoom,reset,save"
)

# Add a line plot and marker scatter points
p.line(x, y, legend_label="sin(x)", line_width=2, color="blue")
p.scatter(x[::10], y[::10], size=8, color="red", alpha=0.5)

# Configure legend behavior
# click_policy='hide' allows users to click the legend to toggle visibility
p.legend.location = "top_left"
p.legend.click_policy = "hide"

output_file("basic_interactive.html")
show(p)

## Subtask 2.2: Adding Hover Tools
### 👶 ELI10: The Magic Magnifying Glass
Hover tools let us see details about a specific data point just by pointing at it. We use a `ColumnDataSource` which is like a organized backpack that carries all our data for Bokeh to use.

In [4]:
import pandas as pd
from bokeh.models import HoverTool, ColumnDataSource
from bokeh.transform import factor_cmap
from bokeh.palettes import Category10

# Create a random dataset using Pandas
np.random.seed(42)
data = pd.DataFrame({
    "x": np.random.randn(100),
    "y": np.random.randn(100),
    "category": np.random.choice(["A", "B", "C"], 100),
    "value": np.random.randint(10, 40, 100)
})

# Convert the DataFrame to a Bokeh ColumnDataSource
source = ColumnDataSource(data)

p = figure(title="Scatter Plot with Hover", width=700, height=450)

# Configure the HoverTool
# @ refers to a column name in our source, $ refers to a special value like coordinate
hover = HoverTool(tooltips=[
    ("Index", "$index"),
    ("(X, Y)", "($x, $y)"),
    ("Category", "@category"),
    ("Value", "@value")
])
p.add_tools(hover)

# Use factor_cmap to color points based on their category label
p.scatter(
    x="x",
    y="y",
    size="value",
    source=source,
    alpha=0.6,
    color=factor_cmap("category", palette=Category10[3], factors=["A", "B", "C"]),
    legend_field="category"
)

show(p)

# Task 3: Working with Stock Data and Tooltips
### 👶 ELI10: Looking at Money Trends
Stocks go up and down every day. We will build a chart that shows these prices and lets you see the exact price and volume for any specific day.

In [5]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# Subtask 3.1: Generate synthetic stock data
def generate_stock_data(symbol="AAPL", days=365):
    """Generates a CSV of fake stock prices."""
    end_date = datetime.now()
    start_date = end_date - timedelta(days=days)
    dates = pd.date_range(start=start_date, end=end_date, freq="D")
    np.random.seed(42)
    prices = [150.0]
    returns = np.random.normal(0.001, 0.02, len(dates))
    for r in returns[1:]: prices.append(max(prices[-1] * (1 + r), 1.0))

    stock_df = pd.DataFrame({
        "date": dates, "symbol": symbol, "price": prices,
        "volume": np.random.randint(1_000_000, 10_000_000, len(dates)),
        "high": [p * (1 + np.random.uniform(0, 0.02)) for p in prices],
        "low": [p * (1 - np.random.uniform(0, 0.02)) for p in prices],
        "change": [0] + [prices[i] - prices[i - 1] for i in range(1, len(prices))]
    })
    return stock_df

stock_data = generate_stock_data()
stock_data.to_csv("stock_data.csv", index=False)
print("Stock data sample:")
display(stock_data.head())

Stock data sample:


,date,symbol,price,volume,high,low,change
0,2025-04-16 14:43:36.252008,AAPL,150.000000,5035493,152.516694,149.679058,0.000000
1,2025-04-17 14:43:36.252008,AAPL,149.735207,6703737,150.108535,147.456154,-0.264793
2,2025-04-18 14:43:36.252008,AAPL,151.824578,2871403,154.620706,150.181026,2.089371
3,2025-04-19 14:43:36.252008,AAPL,156.601070,6747372,159.325604,153.584958,4.776492
4,2025-04-20 14:43:36.252008,AAPL,156.024297,5848287,157.643324,154.957490,-0.576772


In [6]:
from bokeh.models import DatetimeTickFormatter
from bokeh.layouts import column

# Load the data we just created
stock_df = pd.read_csv("stock_data.csv")
stock_df["date"] = pd.to_datetime(stock_df["date"])
source = ColumnDataSource(stock_df)

# Chart 1: Price line chart
price_chart = figure(title="Interactive Stock Price Chart", x_axis_type="datetime", width=900, height=350)
price_chart.line("date", "price", source=source, line_width=2, color="blue", legend_label="Price")

# Add Hover formatting for dates and currency
price_hover = HoverTool(tooltips=[("Date", "@date{%F}"), ("Price", "$@price{0.00}")],
                        formatters={"@date": "datetime"})
price_chart.add_tools(price_hover)

# Chart 2: Volume bar chart (shares price axis x_range for linked zooming)
volume_chart = figure(title="Trading Volume", x_axis_type="datetime", width=900, height=150, x_range=price_chart.x_range)
one_day_ms = 24 * 60 * 60 * 1000
volume_chart.vbar("date", top="volume", source=source, width=0.8 * one_day_ms, color="orange")

# Combine charts into one column layout
layout = column(price_chart, volume_chart)
show(layout)

# Task 4: Implementing Weather Data Visualizations
### 👶 ELI10: Building a Weather Dashboard
A dashboard is a collection of charts that work together to tell a story. Here, we'll see how temperature, rain, and wind interact over a year.

In [7]:
# Subtask 4.1: Generate Weather Data
def generate_weather_data(days=365):
    dates = pd.date_range(start=datetime.now()-timedelta(days=days), periods=days, freq="D")
    temp = 20 + 15 * np.sin(2 * np.pi * (dates.dayofyear - 80) / 365) + np.random.normal(0, 5, days)
    weather_df = pd.DataFrame({
        "date": dates, "temperature": temp, "humidity": np.clip(np.random.normal(60, 15, days), 0, 100),
        "precipitation": np.where(np.random.random(days) < 0.7, 0, np.random.exponential(2, days)),
        "wind_speed": np.random.gamma(2, 3, days),
        "condition": np.random.choice(["Sunny", "Cloudy", "Rainy", "Clear"], days)
    })
    return weather_df

weather_data = generate_weather_data()
weather_data.to_csv("weather_data.csv", index=False)

from bokeh.layouts import gridplot
from bokeh.palettes import RdYlBu11
from bokeh.transform import linear_cmap

# Prep data
weather_df = pd.read_csv("weather_data.csv")
weather_df["date"] = pd.to_datetime(weather_df["date"])
w_source = ColumnDataSource(weather_df)

# Create multiple charts
t_chart = figure(title="Temperature", x_axis_type="datetime", width=400, height=250)
t_chart.scatter("date", "temperature", source=w_source, color=linear_cmap("temperature", RdYlBu11, -10, 40))

h_chart = figure(title="Humidity", x_axis_type="datetime", width=400, height=250)
h_chart.vbar("date", top="humidity", source=w_source, width=0.8 * one_day_ms, color="lightblue")

# Grid layout: 2 rows, 2 columns
grid = gridplot([[t_chart, h_chart]])
show(grid)

# Task 5: Adding Interactive Filters and Sliders
### 👶 ELI10: You're in Control
Now we add buttons and sliders. These use a little bit of JavaScript (the language of the web) so they can update the chart instantly without needing to talk to the server.

In [8]:
from bokeh.models import Slider, Select, CustomJS
from bokeh.layouts import row

# Subtask 5.2: Weather with Sliders and Selectors
weather_df = pd.read_csv("weather_data.csv")
weather_df["plot_y"] = weather_df["temperature"] # Default metric

source = ColumnDataSource(weather_df)
original_source = ColumnDataSource(weather_df)

p = figure(title="Interactive Weather Controls", x_axis_type="datetime", width=600, height=400)
renderer = p.scatter("date", "plot_y", source=source, size=8, color="red", alpha=0.6)

# JavaScript Callback: This code runs in your browser!
callback_code = """
    const data = source.data;
    const orig = original.data;
    const min_temp = slider.value;
    const metric = select.value;

    // Reset data and filter based on slider
    for (const key in orig) { data[key] = []; }
    data["plot_y"] = [];

    for (let i = 0; i < orig.date.length; i++) {
        if (orig.temperature[i] >= min_temp) {
            for (const key in orig) { data[key].push(orig[key][i]); }
            data["plot_y"].push(orig[metric][i]);
        }
    }
    source.change.emit();
"""

temp_slider = Slider(title="Min Temperature", start=0, end=40, value=0, step=1)
metric_select = Select(title="Metric:", value="temperature", options=["temperature", "humidity", "wind_speed"])

# Connect widgets to the callback
callback = CustomJS(args=dict(source=source, original=original_source, slider=temp_slider, select=metric_select), code=callback_code)
temp_slider.js_on_change("value", callback)
metric_select.js_on_change("value", callback)

show(row(column(metric_select, temp_slider), p))

# Task 6: Creating Color Legends and Customizations
### 👶 ELI10: Making it Look Professional
Data isn't just numbers; it's a story. We use colors to highlight differences (like Hot vs. Cold) and add titles and labels to make it clear.

In [9]:
from bokeh.models import ColorBar, LinearColorMapper
from bokeh.palettes import Viridis256

# Create a styled plot
p = figure(title="Professionally Styled Chart", x_axis_type="datetime", width=800, height=400)
p.background_fill_color = "#efefef"
p.grid.grid_line_color = "white"

# Color mapping for temperature
mapper = LinearColorMapper(palette=Viridis256, low=weather_df.temperature.min(), high=weather_df.temperature.max())

p.scatter("date", "temperature", source=w_source,
          color={'field': 'temperature', 'transform': mapper}, size=10)

# Add a color bar legend to the side
color_bar = ColorBar(color_mapper=mapper, label_standoff=12, location=(0,0), title="Temp °C")
p.add_layout(color_bar, 'right')

show(p)

# Task 7: Generating Standalone HTML Output
### 👶 ELI10: Sharing Your Work
You can save these interactive charts as single HTML files. These files can be emailed to a boss or uploaded to a website, and the interactivity still works!

In [10]:
import glob
import os

# We've used output_file throughout the lab.
# Let's list the files we've generated in this Colab session.
html_files = glob.glob("*.html")
print("Stand-alone files ready for download:")
for f in html_files:
    print(f"- {f} ({os.path.getsize(f)} bytes)")

Stand-alone files ready for download:
- test_bokeh_installation.html (6651 bytes)
- basic_interactive.html (31167 bytes)


# ✅ Verification Checklist
- [ ] Bokeh installed and `%pip` successful?
- [ ] `output_notebook()` called?
- [ ] CSV data files generated in the side panel?
- [ ] Charts are interactive (zoom/pan works)?
- [ ] Hover tooltips show correct data?
- [ ] Sliders/Selectors update the charts via `CustomJS`?

# 🛠 Troubleshooting
* **Plot is empty:** Ensure `output_notebook()` was run at the start.
* **Hover tool shows `???`:** Check that the column names in your `HoverTool` match the `ColumnDataSource` keys exactly.
* **Widget doesn't update:** Check the browser console (F12) for JavaScript errors in your `CustomJS` code.

# 📚 Key Takeaways
* **Bokeh** is for web-ready, interactive visualization.
* **ColumnDataSource** is the core data object.
* **CustomJS** enables interactivity without a backend server.
* **Layouts** (row, column, grid) organize multiple charts.

# 🎓 Conclusion
You've successfully built a series of high-quality, interactive visualizations. From simple lines to complex dashboards with JavaScript-powered filters, you now have the skills to turn static data into engaging stories that users can explore.